# Chronos-Bolt Forecast - Sales Demand
Zero-shot forecasting using `amazon/chronos-bolt-small` on retail inventory dataset.

In [8]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [9]:
import pandas as pd
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
from chronos import ChronosBoltPipeline
import os
import glob


MODEL_ID  = "amazon/chronos-bolt-small"
TARGET    = "Demand"
HORIZONS  = [7, 14, 28]
TRAIN_END = "2023-06-30"
VAL_END   = "2023-10-31"
LAG       = 7


# =========================
# Auto detect path
# =========================

if os.path.exists("/kaggle/input"):

    matches = glob.glob(
        "/kaggle/input/**/sales_data.csv",
        recursive=True
    )

    DATA_PATH = matches[0] if matches else "/kaggle/input/sales_data.csv"
    RESULT_DIR = "/kaggle/working"

else:

    DATA_PATH  = "/Users/P837032/Daily/Model/dataset/sales_data.csv"
    RESULT_DIR = "/Users/P837032/Daily/Model/result"


print("DATA_PATH:", DATA_PATH)


# =========================
# Load Chronos model
# =========================

pipeline = ChronosBoltPipeline.from_pretrained(
    MODEL_ID,
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Model loaded")

DATA_PATH: /kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv
Model loaded


## 1. Load Data

In [10]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df["series_id"] = df["Store ID"] + "_" + df["Product ID"]
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} - {df['Date'].max()}")
print(f"Series count: {df['series_id'].nunique()}")
df.head()

Shape: (76000, 17)
Date range: 2022-01-01 00:00:00 - 2024-01-30 00:00:00
Series count: 100


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,series_id
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115,S001_P0001
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229,S001_P0002
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157,S001_P0003
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52,S001_P0004
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59,S001_P0005


## 2. Metrics

In [11]:
def smape(y, p):
    d = np.abs(y) + np.abs(p)
    m = d > 0
    return np.mean(2 * np.abs(y[m] - p[m]) / d[m]) * 100

def mase(y, p, train):
    lag = min(LAG, len(train) - 1)
    denom = np.mean(np.abs(train[lag:] - train[:-lag]))
    return np.mean(np.abs(y - p)) / denom if denom > 0 else np.nan

def rmse(y, p): return np.sqrt(np.mean((y - p) ** 2))
def rmsle(y, p): return np.sqrt(np.mean((np.log1p(np.clip(y,0,None)) - np.log1p(np.clip(p,0,None)))**2))

## 3. Load Model

In [12]:
pipeline = ChronosBoltPipeline.from_pretrained(MODEL_ID, device_map="cpu", torch_dtype=torch.float32)
print('Model loaded.')

Model loaded.


## 4. Rolling Eval per Store × Product × Horizon

In [13]:
df['series_id'] = df['Store ID'] + '_' + df['Product ID']
series_ids = sorted(df['series_id'].unique())
os.makedirs(RESULT_DIR, exist_ok=True)

results = []
details = []

for h in HORIZONS:
    print(f'\n=== Horizon = {h} ===')
    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}

    # Build contexts từ VAL_END trở về trước (dùng làm train context)
    contexts, actuals_list, trains_list = [], [], []
    valid_ids = []
    eval_start = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)

    for sid in series_ids:
        sdf = df[df['series_id'] == sid].sort_values('Date').set_index('Date')
        ts_train = sdf[:VAL_END][TARGET].values.astype(np.float32)
        ts_test  = sdf[eval_start:][TARGET].values.astype(np.float32)
        if len(ts_test) < h or len(ts_train) < 2:
            continue
        contexts.append(torch.tensor(ts_train))
        actuals_list.append(ts_test[:h])
        trains_list.append(ts_train)
        valid_ids.append(sid)

    start = time.time()
    forecasts = pipeline.predict(contexts, prediction_length=h)
    preds = forecasts.median(dim=1).values.numpy()
    print(f'  Inference: {time.time()-start:.1f}s for {len(valid_ids)} series')

    for i, sid in enumerate(valid_ids):
        store, product = sid.split('_', 1)
        y, p, tr = actuals_list[i], preds[i], trains_list[i]
        r = {'smape': smape(y,p), 'mase': mase(y,p,tr), 'rmse': rmse(y,p), 'rmsle': rmsle(y,p)}
        for k in scores: scores[k].append(r[k])
        details.append({'model': 'Chronos-Bolt-Small', 'store': store, 'product': product, 'horizon': h, **{k: round(float(v), 4) for k, v in r.items()}})
        print(f"  {store} | {product} | sMAPE={r['smape']:.2f}% MASE={r['mase']:.4f} RMSE={r['rmse']:.2f} RMSLE={r['rmsle']:.4f}")

    row = {
        'model': 'Chronos-Bolt-Small', 'dataset': 'retail_inventory_daily',
        'target': TARGET, 'horizon': h,
        'mean_smape':   round(float(np.nanmean(scores['smape'])), 4),
        'median_smape': round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':    round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':  round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':    round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':  round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':   round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle': round(float(np.nanmedian(scores['rmsle'])), 4),
    }
    results.append(row)
    print(f"  H={h} | sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSE={row['mean_rmse']:.2f} RMSLE={row['mean_rmsle']:.4f}")

summary = pd.DataFrame(results)
summary


=== Horizon = 7 ===
  Inference: 2.4s for 100 series
  S001 | P0001 | sMAPE=18.70% MASE=0.5955 RMSE=25.32 RMSLE=0.2303
  S001 | P0002 | sMAPE=29.05% MASE=0.5892 RMSE=31.68 RMSLE=0.3221
  S001 | P0003 | sMAPE=19.13% MASE=0.4135 RMSE=25.31 RMSLE=0.2529
  S001 | P0004 | sMAPE=17.37% MASE=0.4359 RMSE=18.72 RMSLE=0.1997
  S001 | P0005 | sMAPE=25.46% MASE=0.5889 RMSE=36.51 RMSLE=0.3287
  S001 | P0006 | sMAPE=26.87% MASE=0.5069 RMSE=23.70 RMSLE=0.2880
  S001 | P0007 | sMAPE=40.27% MASE=1.1844 RMSE=69.03 RMSLE=0.4643
  S001 | P0008 | sMAPE=24.39% MASE=0.6015 RMSE=32.21 RMSLE=0.3122
  S001 | P0009 | sMAPE=31.84% MASE=0.8137 RMSE=42.00 RMSLE=0.3666
  S001 | P0010 | sMAPE=23.88% MASE=0.6170 RMSE=22.03 RMSLE=0.2728
  S001 | P0011 | sMAPE=25.99% MASE=0.4551 RMSE=20.48 RMSLE=0.4021
  S001 | P0012 | sMAPE=16.91% MASE=0.4347 RMSE=29.42 RMSLE=0.2230
  S001 | P0013 | sMAPE=22.83% MASE=0.4930 RMSE=24.68 RMSLE=0.2607
  S001 | P0014 | sMAPE=22.58% MASE=0.5491 RMSE=39.85 RMSLE=0.3016
  S001 | P0015 | sMAPE

,model,dataset,target,horizon,mean_smape,median_smape,mean_mase,median_mase,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,Chronos-Bolt-Small,retail_inventory_daily,Demand,7,29.7727,28.8819,0.7087,0.6739,37.4421,35.5030,0.3697,0.3432
1,Chronos-Bolt-Small,retail_inventory_daily,Demand,14,30.0915,29.6917,0.7159,0.6884,38.1184,37.1219,0.3786,0.3650
2,Chronos-Bolt-Small,retail_inventory_daily,Demand,28,30.9524,30.9804,0.7330,0.7209,39.6030,39.7859,0.3960,0.3823


## 5. Save & Compare

In [14]:
out_path = f'{RESULT_DIR}/chronos_bolt_daily_summary.csv'
summary.to_csv(out_path, index=False)
pd.DataFrame(details).to_csv(f'{RESULT_DIR}/chronos_bolt_daily_details.csv', index=False)
print(f'Saved to {out_path}')

# So sánh với baseline nếu có
baseline_path = f'{RESULT_DIR}/daily_baseline_summary.csv'
if os.path.exists(baseline_path):
    baseline = pd.read_csv(baseline_path)
    comparison = pd.concat([baseline, summary], ignore_index=True)
    display(comparison[['model','horizon','mean_smape','mean_mase','mean_rmse','mean_rmsle']])
else:
    display(summary[['model','horizon','mean_smape','mean_mase','mean_rmse','mean_rmsle']])

Saved to /kaggle/working/chronos_bolt_daily_summary.csv


,model,horizon,mean_smape,mean_mase,mean_rmse,mean_rmsle
0,Chronos-Bolt-Small,7,29.7727,0.7087,37.4421,0.3697
1,Chronos-Bolt-Small,14,30.0915,0.7159,38.1184,0.3786
2,Chronos-Bolt-Small,28,30.9524,0.7330,39.6030,0.3960
